# Part 6: Temporal-LSTM 模型训练及评估

冰川年际序列建模（滑动窗口 LOOKBACK 年 → 预测下一年 dhdt）

评估协议与 02-05 一致：LOYO（Leave-One-Year-Out）留一年交叉验证，有效年份为 2000+LOOKBACK ~ 2019。  
最终模型同时保留 2016 年时间划分测试集结果供参考。

请先运行 `01_dependencies_and_data.ipynb`。

In [ ]:
"""Part 6: Temporal-LSTM. Run 01_dependencies_and_data.ipynb first."""
import os, random, sys
import dill
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

root_dir    = "C:\ML4GM"
outputs_dir = os.path.join(root_dir, "models")

def load_pkl(filepath):
    with open(filepath, "rb") as fr: return dill.load(fr)
def save_pkl(filepath, data):
    with open(filepath, "wb") as fw: dill.dump(data, fw)
    print(f"[{filepath}] data saved.")

_data = load_pkl(os.path.join(outputs_dir, "preprocessed_data.pkl"))
# 强制转为普通 ndarray（防止 memmap 在 Windows loky 子进程中无法访问）
X_all       = np.asarray(_data["X_all"],     dtype=np.float64)
y_all       = np.asarray(_data["y_all"],     dtype=np.float64)
rgiid_all   = np.asarray(_data["rgiid_all"])
year_all    = np.asarray(_data["year_all"])
feature_columns = _data["feature_columns"]

print(f"Loaded: X_all={X_all.shape}, y_all={y_all.shape}, years={sorted(np.unique(year_all))}")
print(f"X_all type: {type(X_all)}, dtype: {X_all.dtype}")


### Temporal-LSTM 数据构建与训练

滑动窗口：连续 LOOKBACK 年的特征 → 预测第 LOOKBACK+1 年的 dhdt  
训练集：目标年 ≤ 2015；测试集：目标年 = 2016（使用 2013-2015 作为上下文，无泄漏）

In [ ]:
def setup_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    torch.use_deterministic_algorithms(True)

setup_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

TRAIN_CUTOFF = 2015   # 训练集目标年上界（含）
TEST_YEAR    = 2016   # 测试集目标年

# 按 (rgiid, year) 排序保证每条冰川时序正确
sort_idx     = np.lexsort((year_all, rgiid_all))
X_sorted     = X_all[sort_idx].astype(np.float32)
y_sorted     = y_all[sort_idx].astype(np.float32)
rgiid_sorted = rgiid_all[sort_idx]
year_sorted  = year_all[sort_idx]


def build_windows(lookback):
    """构建滑动窗口：前 lookback 年为输入，第 lookback+1 年为目标，按时间划分训练/测试集。"""
    WINDOW = lookback + 1
    wx, wy, wyear = [], [], []
    for g in np.unique(rgiid_sorted):
        mask  = rgiid_sorted == g
        Xg    = X_sorted[mask]
        yg    = y_sorted[mask]
        yrs   = year_sorted[mask]
        order = np.argsort(yrs)
        Xg, yg, yrs = Xg[order], yg[order], yrs[order]
        for i in range(len(yrs) - WINDOW + 1):
            wx.append(Xg[i : i + lookback])
            wy.append(float(yg[i + lookback]))
            wyear.append(int(yrs[i + lookback]))
    wx    = np.stack(wx, axis=0)              # [N, lookback, 33]
    wy    = np.array(wy,    dtype=np.float32)
    wyear = np.array(wyear, dtype=int)
    tr_mask = wyear <= TRAIN_CUTOFF
    te_mask = wyear == TEST_YEAR
    X_tr_r, y_tr_ = wx[tr_mask], wy[tr_mask]
    X_te_r, y_te_ = wx[te_mask], wy[te_mask]
    n_tr, L, F = X_tr_r.shape
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr_r.reshape(-1, F)).reshape(n_tr, L, F)
    n_te   = X_te_r.shape[0]
    X_te_s = sc.transform(X_te_r.reshape(-1, F)).reshape(n_te, L, F)
    return X_tr_s, y_tr_, X_te_s, y_te_


print(f"数据已排序，build_windows 函数就绪。TRAIN_CUTOFF={TRAIN_CUTOFF}, TEST_YEAR={TEST_YEAR}")

In [ ]:
class TemporalDataset(Dataset):
    def __init__(self, X_seq, y):
        self.X = torch.FloatTensor(X_seq)
        self.y = torch.FloatTensor(y).unsqueeze(1)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


class TemporalLSTMModel(nn.Module):
    """将连续 LOOKBACK 年特征序列 [B, L, 33] → dhdt"""
    def __init__(self, input_size, hidden=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, num_layers,
                            batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Sequential(
            nn.Linear(hidden, 64), nn.PReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1),
        )
    def forward(self, x):  # x: [B, L, F]
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n[-1])   # last layer hidden state → [B, 1]


def predict_temporal(model, X_seq_np, bs=4096):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_seq_np), bs):
            X_b = torch.FloatTensor(X_seq_np[i:i+bs]).to(device)
            preds.append(model(X_b).cpu().numpy().flatten())
    return np.concatenate(preds)


In [ ]:
import itertools
from sklearn.model_selection import GroupKFold

# ---- Temporal-LSTM 超参数网格搜索（Grid Search，含 LOOKBACK）----
# 网格设计依据：
#   - 每条冰川恰好 20 年（2000-2019），LOOKBACK=2/3/5 分别保留 18/17/15 个窗口
#     窗口总量：8101×(20-LOOKBACK)≈120K-145K，均为大样本
#   - hidden 128/256/512：序列输入维度为 33，较大隐层有利于编码年际变化
#   - 1-2 层 LSTM：年际序列长度短（≤5步），深层LSTM增益有限
#   - dropout/lr/weight_decay 与其他模型保持一致的搜索范围
TUNING_EPOCHS = 50   # 调参阶段快速估算轮数
TUNING_FOLDS  = 2    # 使用 2 折（LOOKBACK 维度使组合数增至 216，折数适当缩减）

param_grid_tlstm = {
    "lookback":      [2, 3, 5],
    "hidden":        [128, 256, 512],
    "num_layers":    [1, 2],
    "dropout":       [0.2, 0.3],
    "learning_rate": [1e-4, 5e-4, 1e-3],
    "weight_decay":  [1e-5, 1e-4],
}
# 3×3×2×2×3×2 = 216 组合

keys   = list(param_grid_tlstm.keys())
combos = list(itertools.product(*param_grid_tlstm.values()))
print(f"[Temporal-LSTM 调参] 网格大小: {len(combos)} 组合 × {TUNING_FOLDS} 折")

best_score, best_params_tlstm = float("-inf"), {}
_cached_windows = {}   # 缓存 build_windows 结果，避免重复构建

for ci, combo in enumerate(combos):
    params = dict(zip(keys, combo))
    lb = params["lookback"]

    if lb not in _cached_windows:
        _cached_windows[lb] = build_windows(lb)
    X_tr_w, y_tr_w, X_te_w, y_te_w = _cached_windows[lb]

    # 时间顺序切分：各折从训练窗口中均匀取连续段作为验证集
    n = len(X_tr_w)
    fold_size = n // (TUNING_FOLDS + 1)

    fold_scores = []
    for fi in range(TUNING_FOLDS):
        va_start = (fi + 1) * fold_size
        va_end   = va_start + fold_size
        va_mask  = np.zeros(n, dtype=bool)
        va_mask[va_start:va_end] = True
        tr_mask  = ~va_mask

        X_tr_f, y_tr_f = X_tr_w[tr_mask], y_tr_w[tr_mask]
        X_va_f, y_va_f = X_tr_w[va_mask], y_tr_w[va_mask]

        setup_seed(42)
        m    = TemporalLSTMModel(X_tr_f.shape[-1],
                                 params["hidden"], params["num_layers"], params["dropout"]).to(device)
        opt_ = torch.optim.Adam(m.parameters(),
                                lr=params["learning_rate"], weight_decay=params["weight_decay"])
        crit_= nn.MSELoss()
        ldr  = DataLoader(TemporalDataset(X_tr_f, y_tr_f),
                          batch_size=256, shuffle=True, num_workers=0)
        for _ in range(TUNING_EPOCHS):
            m.train()
            for X_b, y_b in ldr:
                out  = m(X_b.to(device))
                loss = crit_(out, y_b.to(device))
                opt_.zero_grad(); loss.backward(); opt_.step()
        fold_scores.append(r2_score(y_va_f, predict_temporal(m, X_va_f)))

    mean_r2 = float(np.mean(fold_scores))
    print(f"  [{ci+1:3d}/{len(combos)}] lb={lb} hidden={params['hidden']} "
          f"layers={params['num_layers']} drop={params['dropout']:.1f} "
          f"lr={params['learning_rate']:.0e} wd={params['weight_decay']:.0e} → R²={mean_r2:.4f}")
    if mean_r2 > best_score:
        best_score        = mean_r2
        best_params_tlstm = params

LOOKBACK         = best_params_tlstm["lookback"]
lstm_hidden      = best_params_tlstm["hidden"]
lstm_layers      = best_params_tlstm["num_layers"]
dropout          = best_params_tlstm["dropout"]
learning_rate    = best_params_tlstm["learning_rate"]
weight_decay_val = best_params_tlstm["weight_decay"]
batch_size       = 512
epochs           = 300
patience         = 30
print(f"\n[Temporal-LSTM 调参] Best R²={best_score:.4f}")
print(f"  LOOKBACK={LOOKBACK}, hidden={lstm_hidden}, layers={lstm_layers}, "
      f"dropout={dropout:.2f}, lr={learning_rate:.2e}, weight_decay={weight_decay_val:.2e}")

# 用最优 LOOKBACK 重新构建最终训练/测试数据
X_tr_std, y_tr, X_te_std, y_te = build_windows(LOOKBACK)
print(f"最终数据: train={X_tr_std.shape}, test={X_te_std.shape}")

# 90/10 train/val split（来自训练窗口，用于 early stopping）
n_tr_total = len(X_tr_std)
rng    = np.random.RandomState(42)
va_idx = rng.choice(n_tr_total, size=int(n_tr_total * 0.1), replace=False)
tr_idx = np.setdiff1d(np.arange(n_tr_total), va_idx)

tr_loader = DataLoader(TemporalDataset(X_tr_std[tr_idx], y_tr[tr_idx]),
                       batch_size=batch_size, shuffle=True,  num_workers=0)
va_loader = DataLoader(TemporalDataset(X_tr_std[va_idx], y_tr[va_idx]),
                       batch_size=batch_size, shuffle=False, num_workers=0)

In [ ]:
best_val_r2, patience_counter = float("-inf"), 0
train_r2_records, train_loss_records = [], []
val_r2_records,   val_loss_records   = [], []

setup_seed(42)
temporal_lstm = TemporalLSTMModel(
    input_size=X_tr_std.shape[-1],
    hidden=lstm_hidden, num_layers=lstm_layers, dropout=dropout
).to(device)
optimizer  = torch.optim.Adam(temporal_lstm.parameters(), lr=learning_rate, weight_decay=weight_decay_val)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
criterion  = nn.MSELoss()

for epoch in range(1, epochs + 1):
    temporal_lstm.train()
    tr_real, tr_pred, tr_losses = [], [], []
    for X_b, y_b in tr_loader:
        out  = temporal_lstm(X_b.to(device))
        loss = criterion(out, y_b.to(device))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tr_real.extend(y_b.numpy().flatten()); tr_pred.extend(out.detach().cpu().numpy().flatten())
        tr_losses.append(loss.item())
    tr_r2   = round(r2_score(tr_real, tr_pred), 4)
    tr_loss = round(float(np.mean(tr_losses)), 6)

    temporal_lstm.eval()
    va_real, va_pred, va_losses = [], [], []
    with torch.no_grad():
        for X_b, y_b in va_loader:
            out  = temporal_lstm(X_b.to(device))
            loss = criterion(out, y_b.to(device))
            va_real.extend(y_b.numpy().flatten()); va_pred.extend(out.cpu().numpy().flatten())
            va_losses.append(loss.item())
    va_r2   = round(r2_score(va_real, va_pred), 4)
    va_loss = round(float(np.mean(va_losses)), 6)
    scheduler.step(va_loss)

    print(f"Epoch {epoch}/{epochs} | Train R2={tr_r2:.4f} Loss={tr_loss:.6f} | "
          f"Val R2={va_r2:.4f} Loss={va_loss:.6f}")
    train_r2_records.append(tr_r2);  train_loss_records.append(tr_loss)
    val_r2_records.append(va_r2);    val_loss_records.append(va_loss)

    if va_r2 > best_val_r2:
        best_val_r2 = va_r2; patience_counter = 0
        torch.save(temporal_lstm.state_dict(), os.path.join(outputs_dir, "temporal_lstm_best.pt"))
    else:
        patience_counter += 1
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch}"); break

print(f"Done! Best Val R2: {best_val_r2:.4f}")

In [ ]:
temporal_lstm.load_state_dict(
    torch.load(os.path.join(outputs_dir, "temporal_lstm_best.pt"), map_location=device))
temporal_lstm.eval()
save_pkl(os.path.join(outputs_dir, "temporal_lstm_model.pkl"), temporal_lstm)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)
axes[0].plot(train_r2_records, label="Train R2")
axes[0].plot(val_r2_records,   label="Val R2")
axes[0].set_title("R2 during training (Temporal-LSTM)"); axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("R2"); axes[0].legend()
axes[1].plot(train_loss_records, label="Train Loss")
axes[1].plot(val_loss_records,   label="Val Loss")
axes[1].set_title("Loss during training (Temporal-LSTM)"); axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss"); axes[1].legend()
plt.tight_layout(); plt.show(); plt.close()


In [ ]:
# 训练集指标（目标年 ≤ 2015）
y_train_pred_of_temporal_lstm = predict_temporal(temporal_lstm, X_tr_std)
y_train_of_temporal_lstm      = y_tr.tolist()
y_train_pred_of_temporal_lstm = y_train_pred_of_temporal_lstm.tolist()

print(f"Temporal-LSTM - Train (target year ≤ {TRAIN_CUTOFF}):")
print(f"MAE:  {mean_absolute_error(y_train_of_temporal_lstm, y_train_pred_of_temporal_lstm):.4f}")
print(f"MSE:  {mean_squared_error(y_train_of_temporal_lstm, y_train_pred_of_temporal_lstm):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_train_of_temporal_lstm, y_train_pred_of_temporal_lstm)):.4f}")
print(f"R2:   {r2_score(y_train_of_temporal_lstm, y_train_pred_of_temporal_lstm):.4f}")


In [ ]:
# 测试集指标（目标年 = 2016，上下文 2013-2015，无泄漏）
y_test_pred_of_temporal_lstm = predict_temporal(temporal_lstm, X_te_std)
y_test_of_temporal_lstm      = y_te.tolist()
y_test_pred_of_temporal_lstm = y_test_pred_of_temporal_lstm.tolist()

print(f"Temporal-LSTM - Test (target year = {TEST_YEAR}, context={TEST_YEAR-LOOKBACK}–{TEST_YEAR-1}):")
print(f"MAE:  {mean_absolute_error(y_test_of_temporal_lstm, y_test_pred_of_temporal_lstm):.4f}")
print(f"MSE:  {mean_squared_error(y_test_of_temporal_lstm, y_test_pred_of_temporal_lstm):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_of_temporal_lstm, y_test_pred_of_temporal_lstm)):.4f}")
print(f"R2:   {r2_score(y_test_of_temporal_lstm, y_test_pred_of_temporal_lstm):.4f}")

save_pkl(os.path.join(outputs_dir, "temporal_lstm_test_pred.pkl"), y_test_pred_of_temporal_lstm)


In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(12, 7), dpi=100)
ax.plot(y_test_of_temporal_lstm,      linewidth=2, label="Real values")
ax.plot(y_test_pred_of_temporal_lstm, linewidth=2, label="Predict values")
ax.set_title(f"Temporal-LSTM — Test Year {TEST_YEAR} (context {TEST_YEAR-LOOKBACK}–{TEST_YEAR-1})",
             fontsize=18)
ax.set_xlabel("Glacier index", fontsize=14); ax.set_ylabel("dhdt (m/yr)", fontsize=14)
ax.tick_params(labelsize=12); ax.legend(loc="best", prop={"size": 14})
plt.show(); plt.close()


In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(12, 10), dpi=100)
ax.text(
    min(y_test_of_temporal_lstm), max(y_test_of_temporal_lstm),
    f"$MAE={round(mean_absolute_error(y_test_of_temporal_lstm, y_test_pred_of_temporal_lstm), 4)}$"
    f"\n$MSE={round(mean_squared_error(y_test_of_temporal_lstm, y_test_pred_of_temporal_lstm), 4)}$"
    f"\n$RMSE={round(pow(mean_squared_error(y_test_of_temporal_lstm, y_test_pred_of_temporal_lstm), 0.5), 4)}$"
    f"\n$R^2={round(r2_score(y_test_of_temporal_lstm, y_test_pred_of_temporal_lstm), 4)}$",
    verticalalignment="top", fontdict={"size": 14, "color": "k"},
)
ax.scatter(y_test_of_temporal_lstm, y_test_pred_of_temporal_lstm,
           c="none", marker="o", edgecolors="k")
from sklearn.linear_model import LinearRegression
fm = LinearRegression()
fm.fit([[v] for v in y_test_of_temporal_lstm], y_test_pred_of_temporal_lstm)
ax.plot([min(y_test_of_temporal_lstm), max(y_test_of_temporal_lstm)],
        [fm.predict([[min(y_test_of_temporal_lstm)]]).item(),
         fm.predict([[max(y_test_of_temporal_lstm)]]).item()],
        linewidth=2, linestyle="--", color="r", label="Fitting curve")
ax.plot([min(y_test_of_temporal_lstm), max(y_test_of_temporal_lstm)],
        [min(y_test_of_temporal_lstm), max(y_test_of_temporal_lstm)],
        linewidth=2, linestyle="-",  color="r", label="Reference curve")
ax.set_title(f"Temporal-LSTM Residual (Test Year {TEST_YEAR})", fontsize=20)
ax.set_xlabel("Real values", fontsize=14); ax.set_ylabel("Predict values", fontsize=14)
ax.tick_params(labelsize=12); ax.legend(loc="lower right", prop={"size": 14})
plt.show(); plt.close()


### LOYO 留一年交叉验证（Leave-One-Year-Out）

使用最优超参数（含 LOOKBACK），对每个有效目标年依次留出，其余年份训练，评估留出年的 R²/RMSE/MAE。  
有效年份：2000+LOOKBACK ~ 2019（早期年份因缺少足够历史窗口而跳过）。  
结果保存为 `temporal_lstm_loyo_results.csv`，供 07_compare.ipynb 统一对比。

In [ ]:
# ---- Temporal-LSTM LOYO 留一年交叉验证 ----
# 使用最优超参数（LOOKBACK/hidden/layers/dropout/lr/wd），逐年留出评估
LOYO_EPOCHS   = 200   # 每折最大训练轮数（early stopping 通常提前结束）
LOYO_PATIENCE = 20    # early stopping patience

loyo_years = list(range(2000 + LOOKBACK, 2020))   # 有效目标年
print(f"[Temporal-LSTM LOYO] LOOKBACK={LOOKBACK}，评估年份: {loyo_years}（共 {len(loyo_years)} fold）")


def build_windows_loyo(lookback, hold_out_year):
    """LOYO 版本：训练集 = 除 hold_out_year 以外所有目标年，测试集 = hold_out_year。"""
    WINDOW = lookback + 1
    wx, wy, wyear = [], [], []
    for g in np.unique(rgiid_sorted):
        mask  = rgiid_sorted == g
        Xg    = X_sorted[mask]; yg = y_sorted[mask]; yrs = year_sorted[mask]
        order = np.argsort(yrs)
        Xg, yg, yrs = Xg[order], yg[order], yrs[order]
        for i in range(len(yrs) - WINDOW + 1):
            wx.append(Xg[i : i + lookback])
            wy.append(float(yg[i + lookback]))
            wyear.append(int(yrs[i + lookback]))
    wx    = np.stack(wx, axis=0)
    wy    = np.array(wy,    dtype=np.float32)
    wyear = np.array(wyear, dtype=int)
    tr_mask = wyear != hold_out_year
    te_mask = wyear == hold_out_year
    X_tr_r, y_tr_ = wx[tr_mask], wy[tr_mask]
    X_te_r, y_te_ = wx[te_mask], wy[te_mask]
    if len(X_te_r) == 0:
        return None
    n_tr, L, F = X_tr_r.shape
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr_r.reshape(-1, F)).reshape(n_tr, L, F)
    n_te   = X_te_r.shape[0]
    X_te_s = sc.transform(X_te_r.reshape(-1, F)).reshape(n_te, L, F)
    return X_tr_s, y_tr_, X_te_s, y_te_


loyo_records = []
all_y_true_loyo, all_y_pred_loyo = [], []   # OOF 收集
for yi, yr in enumerate(loyo_years):
    result = build_windows_loyo(LOOKBACK, yr)
    if result is None:
        print(f"  [SKIP] year {yr}: no test windows")
        continue
    X_tr_l, y_tr_l, X_te_l, y_te_l = result

    # 10% val split for early stopping
    n_tr_l   = len(X_tr_l)
    rng_l    = np.random.RandomState(42)
    va_idx_l = rng_l.choice(n_tr_l, size=int(n_tr_l * 0.1), replace=False)
    tr_idx_l = np.setdiff1d(np.arange(n_tr_l), va_idx_l)

    tr_ldr_l = DataLoader(TemporalDataset(X_tr_l[tr_idx_l], y_tr_l[tr_idx_l]),
                          batch_size=512, shuffle=True,  num_workers=0)

    setup_seed(42)
    m_loyo = TemporalLSTMModel(X_tr_l.shape[-1], lstm_hidden, lstm_layers, dropout).to(device)
    opt_l  = torch.optim.Adam(m_loyo.parameters(), lr=learning_rate, weight_decay=weight_decay_val)
    sch_l  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_l, patience=5, factor=0.5)
    crit_l = nn.MSELoss()

    best_va_r2_l, pat_l, best_state_l = float("-inf"), 0, None
    for ep in range(1, LOYO_EPOCHS + 1):
        m_loyo.train()
        for X_b, y_b in tr_ldr_l:
            out  = m_loyo(X_b.to(device))
            loss = crit_l(out, y_b.to(device))
            opt_l.zero_grad(); loss.backward(); opt_l.step()
        m_loyo.eval()
        va_preds = predict_temporal(m_loyo, X_tr_l[va_idx_l])
        va_r2_l  = r2_score(y_tr_l[va_idx_l], va_preds)
        sch_l.step(-va_r2_l)
        if va_r2_l > best_va_r2_l:
            best_va_r2_l = va_r2_l; pat_l = 0
            best_state_l = {k: v.cpu().clone() for k, v in m_loyo.state_dict().items()}
        else:
            pat_l += 1
        if pat_l >= LOYO_PATIENCE:
            break

    m_loyo.load_state_dict(best_state_l)
    y_pred_l = predict_temporal(m_loyo, X_te_l)

    all_y_true_loyo.extend(y_te_l.tolist())
    all_y_pred_loyo.extend(y_pred_l.tolist())

    r2_l     = r2_score(y_te_l, y_pred_l)
    rmse_l   = float(np.sqrt(mean_squared_error(y_te_l, y_pred_l)))
    mae_l    = float(mean_absolute_error(y_te_l, y_pred_l))
    loyo_records.append({"year": yr, "R2": round(r2_l, 4), "RMSE": round(rmse_l, 4), "MAE": round(mae_l, 4)})
    print(f"  [{yi+1:2d}/{len(loyo_years)}] year={yr} | R²={r2_l:.4f}  RMSE={rmse_l:.4f}  MAE={mae_l:.4f}")

loyo_df_tlstm = pd.DataFrame(loyo_records)
csv_path_l    = os.path.join(outputs_dir, "temporal_lstm_loyo_results.csv")
loyo_df_tlstm.to_csv(csv_path_l, index=False)
print(f"\n[Temporal-LSTM LOYO] 结果已保存: {csv_path_l}")
print(f"平均 R²={loyo_df_tlstm['R2'].mean():.4f}  RMSE={loyo_df_tlstm['RMSE'].mean():.4f}  MAE={loyo_df_tlstm['MAE'].mean():.4f}")
print(loyo_df_tlstm.to_string(index=False))

# ---- 逐年 R²/RMSE 折线图 ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)
axes[0].plot(loyo_df_tlstm["year"], loyo_df_tlstm["R2"],   marker="o", linewidth=2)
axes[0].set_title("Temporal-LSTM LOYO — Per-Year R²")
axes[0].set_xlabel("Held-out Year"); axes[0].set_ylabel("R²"); axes[0].grid(True, alpha=0.3)
axes[1].plot(loyo_df_tlstm["year"], loyo_df_tlstm["RMSE"], marker="s", linewidth=2, color="orange")
axes[1].set_title("Temporal-LSTM LOYO — Per-Year RMSE")
axes[1].set_xlabel("Held-out Year"); axes[1].set_ylabel("RMSE (m/yr)"); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show(); plt.close()

# ---- OOF 散点图（Bolibar et al. 2020 Figure 9 风格） ----
all_y_true_np = np.array(all_y_true_loyo)
all_y_pred_np = np.array(all_y_pred_loyo)
r2_oof   = float(r2_score(all_y_true_np, all_y_pred_np))
rmse_oof = float(np.sqrt(mean_squared_error(all_y_true_np, all_y_pred_np)))
mae_oof  = float(mean_absolute_error(all_y_true_np, all_y_pred_np))

fig, ax = plt.subplots(figsize=(7, 7), dpi=100)
ax.scatter(all_y_true_np, all_y_pred_np, alpha=0.15, s=8, c="steelblue")
lims = [float(all_y_true_np.min()), float(all_y_true_np.max())]
ax.plot(lims, lims, "r--", linewidth=1.5)
ax.set_xlabel("Observed dhdt (m/yr)", fontsize=13)
ax.set_ylabel("Predicted dhdt (m/yr)", fontsize=13)
ax.set_title(f"LOYO Out-of-Fold Predictions (Temporal-LSTM)\n"
             f"R²={r2_oof:.4f}  RMSE={rmse_oof:.4f}  MAE={mae_oof:.4f}", fontsize=13)
plt.tight_layout()
plt.show()
plt.close()
print(f"[LOYO-TLSTM OOF] R²={r2_oof:.4f}  RMSE={rmse_oof:.4f}  MAE={mae_oof:.4f}  n={len(all_y_true_np)}")

### 方案一：5-fold 空间 GroupKFold CV

按**目标冰川 ID**（`w_rgiid`）分组（GroupKFold），每折 80% 冰川作为训练集，20% 冰川作为测试集。  
**分组键为窗口目标冰川，而非输入历史年份，保证同一冰川的所有窗口不会同时出现在训练集和测试集中。**  
每折独立拟合 StandardScaler（对窗口展平后拟合），从训练集再拨 10% 用于 early stopping（patience=20，最多 200 轮）。

In [ ]:
from sklearn.model_selection import KFold

CV_EPOCHS   = 200
CV_PATIENCE = 20

# ── 辅助函数：冰川/年份折编号 ──────────────────────────────────────────
def make_glacier_fold(rgiid_arr, n_folds=5):
    glacier_sorted = np.sort(np.unique(rgiid_arr))
    n_glaciers = len(glacier_sorted)
    gid_to_fold = {gid: int(i * n_folds // n_glaciers)
                   for i, gid in enumerate(glacier_sorted)}
    return np.array([gid_to_fold[g] for g in rgiid_arr], dtype=int)

def make_year_fold(year_arr, n_folds=5):
    year_sorted = np.sort(np.unique(year_arr))
    n_years = len(year_sorted)
    yr_to_fold = {yr: int(i * n_folds // n_years)
                  for i, yr in enumerate(year_sorted)}
    return np.array([yr_to_fold[y] for y in year_arr], dtype=int)


def build_windows_for_cv(lookback):
    """构建全量滑动窗口（不限制目标年），同时返回目标冰川 ID 和目标年供 CV 分组。"""
    WINDOW = lookback + 1
    wx, wy, w_rgiid, w_year = [], [], [], []
    for g in np.unique(rgiid_sorted):
        mask = rgiid_sorted == g
        Xg = X_sorted[mask]; yg = y_sorted[mask]; yrs = year_sorted[mask]
        order = np.argsort(yrs)
        Xg, yg, yrs = Xg[order], yg[order], yrs[order]
        for i in range(len(yrs) - WINDOW + 1):
            wx.append(Xg[i : i + lookback])
            wy.append(float(yg[i + lookback]))
            w_rgiid.append(g)
            w_year.append(int(yrs[i + lookback]))
    wx      = np.stack(wx, axis=0)                    # [N, lookback, 33]
    wy      = np.array(wy,      dtype=np.float32)
    w_rgiid = np.array(w_rgiid)
    w_year  = np.array(w_year,  dtype=int)
    return wx, wy, w_rgiid, w_year


def train_fold_tlstm(X_tr_s, y_tr, X_te_s, y_te, fold_label):
    """训练一折 Temporal-LSTM，返回 (r2, rmse, mae)"""
    n_tr = len(X_tr_s)
    rng_ = np.random.RandomState(42)
    va_i = rng_.choice(n_tr, size=int(n_tr * 0.1), replace=False)
    tr_i = np.setdiff1d(np.arange(n_tr), va_i)

    tr_ldr = DataLoader(TemporalDataset(X_tr_s[tr_i], y_tr[tr_i]),
                        batch_size=512, shuffle=True, num_workers=0)
    setup_seed(42)
    m    = TemporalLSTMModel(X_tr_s.shape[-1], lstm_hidden, lstm_layers, dropout).to(device)
    opt_ = torch.optim.Adam(m.parameters(), lr=learning_rate, weight_decay=weight_decay_val)
    sch_ = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_, patience=5, factor=0.5)
    crit_= nn.MSELoss()

    best_va, pat_, best_st = float("-inf"), 0, None
    for ep in range(1, CV_EPOCHS + 1):
        m.train()
        for X_b, y_b in tr_ldr:
            out  = m(X_b.to(device))
            loss = crit_(out, y_b.to(device))
            opt_.zero_grad(); loss.backward(); opt_.step()
        va_preds = predict_temporal(m, X_tr_s[va_i])
        va_r2    = r2_score(y_tr[va_i], va_preds)
        sch_.step(-va_r2)
        if va_r2 > best_va:
            best_va = va_r2; pat_ = 0
            best_st = {k: v.cpu().clone() for k, v in m.state_dict().items()}
        else:
            pat_ += 1
        if pat_ >= CV_PATIENCE:
            print(f"  {fold_label}: early stop @ epoch {ep}, val R²={best_va:.4f}")
            break
    m.load_state_dict(best_st)
    y_pred = predict_temporal(m, X_te_s)
    r2   = float(r2_score(y_te, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_te, y_pred)))
    mae  = float(mean_absolute_error(y_te, y_pred))
    return r2, rmse, mae


# ── 构建全量 CV 窗口 ───────────────────────────────────────────────────
wx_all_cv, wy_all_cv, w_rgiid_cv, w_year_cv = build_windows_for_cv(LOOKBACK)
n_cv, L_cv, F_cv = wx_all_cv.shape
print(f"CV 全量窗口: {wx_all_cv.shape}")

# ── 5-fold 随机 KFold CV ──────────────────────────────────────────────
kf_cv = KFold(n_splits=5, shuffle=True, random_state=42)
kfold_rows = []

for fold_i, (tr_idx, te_idx) in enumerate(kf_cv.split(wx_all_cv, wy_all_cv), 1):
    X_tr_raw = wx_all_cv[tr_idx]; y_tr_ = wy_all_cv[tr_idx]
    X_te_raw = wx_all_cv[te_idx]; y_te_ = wy_all_cv[te_idx]
    n_tr_f   = len(X_tr_raw)

    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr_raw.reshape(-1, F_cv)).reshape(n_tr_f, L_cv, F_cv)
    X_te_s = sc.transform(X_te_raw.reshape(-1, F_cv)).reshape(len(X_te_raw), L_cv, F_cv)

    print(f"\n[TLSTM KFold CV] Fold {fold_i}/5 | n_train={n_tr_f} | n_test={len(te_idx)}")
    r2, rmse, mae = train_fold_tlstm(X_tr_s, y_tr_, X_te_s, y_te_, f"Fold{fold_i}")

    kfold_rows.append({
        "fold": fold_i, "n_train": n_tr_f, "n_test": len(te_idx),
        "R2": r2, "RMSE": rmse, "MAE": mae,
    })
    print(f"[TLSTM KFold CV] Fold {fold_i}/5 | R2={r2:.4f} | RMSE={rmse:.4f} | MAE={mae:.4f}")

kfold_df = pd.DataFrame(kfold_rows)
kf_path = os.path.join(outputs_dir, "temporal_lstm_kfold_cv_results.csv")
kfold_df.to_csv(kf_path, index=False)

print(f"\n[TLSTM KFold CV] ===== Summary =====")
print(f"  R²   mean={kfold_df['R2'].mean():.4f}  std={kfold_df['R2'].std():.4f}")
print(f"  RMSE mean={kfold_df['RMSE'].mean():.4f}  std={kfold_df['RMSE'].std():.4f}")
print(f"  MAE  mean={kfold_df['MAE'].mean():.4f}  std={kfold_df['MAE'].std():.4f}")
print(f"  Saved → {kf_path}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=100)
for ax, metric, color in zip(axes, ["R2", "RMSE", "MAE"],
                              ["steelblue", "darkorange", "seagreen"]):
    vals = kfold_df[metric]
    ax.bar(kfold_df["fold"], vals, color=color, alpha=0.8)
    ax.axhline(vals.mean(), color="r", linestyle="--", label=f"Mean={vals.mean():.4f}")
    ax.set_xlabel("Fold", fontsize=12); ax.set_ylabel(metric, fontsize=12)
    ax.set_title(f"Temporal-LSTM KFold CV — {metric}", fontsize=13); ax.legend()
plt.tight_layout(); plt.show(); plt.close()


### 方案二：5×5 Block CV（对角线）

目标冰川按字典序分 5 组，目标年按时间顺序分 5 组，对角线块（冰川组 i ∩ 年份组 i）轮流作为测试集。  
训练集排除测试块所在的**整行**（同冰川组所有目标年）和**整列**（同目标年组所有冰川），约占 64%。  
**分组键均基于窗口目标冰川/目标年，同时消除空间泄露和时间泄露。**

In [ ]:
# ── 5×5 Block CV（对角线） ────────────────────────────────────────────
glacier_fold_cv = make_glacier_fold(w_rgiid_cv, n_folds=5)
year_fold_cv    = make_year_fold(w_year_cv,    n_folds=5)

block_rows = []

for f in range(5):
    test_mask  = (glacier_fold_cv == f) & (year_fold_cv == f)
    train_mask = ~((glacier_fold_cv == f) | (year_fold_cv == f))

    assert not np.any(glacier_fold_cv[train_mask] == f), \
        f"Block fold {f}: glacier leakage in train!"
    assert not np.any(year_fold_cv[train_mask] == f), \
        f"Block fold {f}: year leakage in train!"

    if test_mask.sum() == 0:
        print(f"[TLSTM Block CV] Fold {f}: empty test set, skipping.")
        continue

    X_tr_raw = wx_all_cv[train_mask]; y_tr_ = wy_all_cv[train_mask]
    X_te_raw = wx_all_cv[test_mask];  y_te_ = wy_all_cv[test_mask]
    n_tr_f   = len(X_tr_raw)

    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr_raw.reshape(-1, F_cv)).reshape(n_tr_f, L_cv, F_cv)
    X_te_s = sc.transform(X_te_raw.reshape(-1, F_cv)).reshape(len(X_te_raw), L_cv, F_cv)

    print(f"\n[TLSTM Block CV] Fold {f}/4 | n_train={n_tr_f} | n_test={test_mask.sum()}")
    r2, rmse, mae = train_fold_tlstm(X_tr_s, y_tr_, X_te_s, y_te_, f"Block{f}")
    n_test_glaciers = len(np.unique(w_rgiid_cv[test_mask]))

    block_rows.append({
        "fold": f, "n_train": n_tr_f, "n_test": int(test_mask.sum()),
        "n_test_glaciers": n_test_glaciers, "R2": r2, "RMSE": rmse, "MAE": mae,
    })
    print(f"[TLSTM Block CV] Fold {f}/4 | glaciers={n_test_glaciers} | "
          f"R2={r2:.4f} | RMSE={rmse:.4f} | MAE={mae:.4f}")

block_df = pd.DataFrame(block_rows)
blk_path = os.path.join(outputs_dir, "temporal_lstm_block_cv_results.csv")
block_df.to_csv(blk_path, index=False)

print(f"\n[TLSTM Block CV] ===== Summary =====")
print(f"  R²   mean={block_df['R2'].mean():.4f}  std={block_df['R2'].std():.4f}")
print(f"  RMSE mean={block_df['RMSE'].mean():.4f}  std={block_df['RMSE'].std():.4f}")
print(f"  MAE  mean={block_df['MAE'].mean():.4f}  std={block_df['MAE'].std():.4f}")
print(f"  Saved → {blk_path}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=100)
for ax, metric, color in zip(axes, ["R2", "RMSE", "MAE"],
                              ["steelblue", "darkorange", "seagreen"]):
    vals = block_df[metric]
    ax.bar(block_df["fold"], vals, color=color, alpha=0.8)
    ax.axhline(vals.mean(), color="r", linestyle="--", label=f"Mean={vals.mean():.4f}")
    ax.set_xlabel("Fold", fontsize=12); ax.set_ylabel(metric, fontsize=12)
    ax.set_title(f"Temporal-LSTM Block CV — {metric}", fontsize=13); ax.legend()
plt.tight_layout(); plt.show(); plt.close()